# ML baseline — Fraud & Collusion Detection in Public Procurement

**Owner:** ML module. **Goal of this notebook:** answer EDA question 6 of the README ("how do the screens relate to the label?") and get a first honest baseline.

Questions it answers:
1. Which pre-computed screens separate `is_cartel = 1` from `0`?
2. How well does a simple model do when validated **by cartel case** (`GroupKFold`)?
3. How much of a random-split score is leakage? (random vs group split)
4. How does it behave on the largest case and on a country it has never seen?

**Run it from the repo root or from `notebooks/`.** Needs: `pip install scikit-learn xgboost matplotlib shap`.

Reminder from the dataset findings: the target is `is_cartel`; PR-AUC of a random model is about 0.42; `cartel_id`, `cartel_tender`, `bidder_id`, `buyer_id` and other IDs are never features.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import GroupKFold, StratifiedKFold, LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             precision_score, recall_score, f1_score, accuracy_score)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
N_FOLDS = 5
FILENAME = "GTI_labelled_cartel_data_NOV2023.csv"
CANDIDATE_PATHS = [os.path.join("data", "raw", FILENAME), os.path.join("..", "data", "raw", FILENAME)]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
assert DATA_PATH, f"CSV not found. Looked in: {CANDIDATE_PATHS}"
print("Using:", DATA_PATH)

## 1. Load and sanity checks

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
TARGET, GROUP = "is_cartel", "cartel_id"
y = df[TARGET].astype(int).values
groups = df[GROUP].values

sizes = pd.Series(groups).value_counts()
print("Shape:", df.shape)
print(f"Positive rate: {y.mean():.3f}  -> a random model scores PR-AUC ~ {y.mean():.2f}")
print("Cartel cases:", len(sizes), "| largest case share of rows:", round(sizes.iloc[0] / len(df), 3))

## 2. Feature sets

Rules (from the dataset findings): never use the target, `cartel_id`, `cartel_tender` (identifier-like) or raw IDs.
`tender_year` is kept out of the baseline because years can act as a proxy for the cartel period. We test it separately.

In [ ]:
BANNED = ["is_cartel", "cartel_id", "cartel_tender", "persistent_id", "tender_id", "lot_id", "bidder_id", "buyer_id"]

numeric_cols = [c for c in df.select_dtypes(include="number").columns if c not in BANNED]
numeric_no_year = [c for c in numeric_cols if c != "tender_year"]

cat = pd.get_dummies(df[["cpv2_smp", "country"]].astype(str), prefix=["cpv2", "country"], dtype=float)

X_sets = {
    "screens": df[numeric_no_year].astype(float),
    "screens + year": df[numeric_cols].astype(float),
    "screens + sector + country": pd.concat([df[numeric_no_year].astype(float), cat], axis=1),
}
for name, X in X_sets.items():
    assert not set(BANNED) & set(X.columns), f"banned column in {name}"
    print(f"{name:30s} {X.shape[1]:3d} features | NaN cells: {int(X.isna().sum().sum())}")
print()
print("Screens used:", numeric_no_year)

## 3. How does each screen relate to the label?

`strength` = how well one feature alone ranks cartel vs other contracts (0.5 = useless, 1.0 = perfect).
`strength_case_weighted` gives every cartel case the same total weight, so a few huge cases cannot dominate.
If a screen is strong overall but weak case-weighted, it is probably describing a big case, not collusion in general.

In [ ]:
case_weight = 1.0 / pd.Series(groups).map(sizes).values
rows = []
for c in X_sets["screens"].columns:
    x = X_sets["screens"][c].values
    if np.nanstd(x) == 0:
        continue
    x = np.nan_to_num(x, nan=np.nanmedian(x))
    auc = roc_auc_score(y, x)
    auc_w = roc_auc_score(y, x, sample_weight=case_weight)
    rows.append({"feature": c, "direction": "+" if auc >= 0.5 else "-",
                 "strength": max(auc, 1 - auc), "strength_case_weighted": max(auc_w, 1 - auc_w),
                 "mean_cartel": x[y == 1].mean(), "mean_other": x[y == 0].mean()})
uni = pd.DataFrame(rows).sort_values("strength", ascending=False).reset_index(drop=True)
display(uni.round(3))

## 4. Validation helper

Contracts of the same cartel are strongly correlated, so every score below is computed with **`GroupKFold` by `cartel_id`**: a cartel is either entirely in train or entirely in test.
Threshold-free PR-AUC is the main metric; precision/recall/F1 use a fixed 0.5 threshold for now (choosing a threshold properly comes later, on out-of-fold predictions).

In [ ]:
def run_cv(make_model, X, y, groups, splitter, threshold=0.5):
    X = X.reset_index(drop=True)
    oof = np.full(len(y), np.nan)
    folds = []
    for i, (tr, te) in enumerate(splitter.split(X, y, groups), start=1):
        model = make_model()
        model.fit(X.iloc[tr], y[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        oof[te] = p
        pred = (p >= threshold).astype(int)
        has_both = len(np.unique(y[te])) == 2
        names = np.unique(groups[te])
        folds.append({
            "fold": i, "n_test": len(te), "test_groups": ",".join(map(str, names[:3])) + ("..." if len(names) > 3 else ""),
            "pos_rate": y[te].mean(),
            "pr_auc": average_precision_score(y[te], p),
            "roc_auc": roc_auc_score(y[te], p) if has_both else np.nan,
            "accuracy": accuracy_score(y[te], pred),
            "precision": precision_score(y[te], pred, zero_division=0),
            "recall": recall_score(y[te], pred, zero_division=0),
            "f1": f1_score(y[te], pred, zero_division=0),
        })
    return pd.DataFrame(folds), oof


def make_dummy():
    return DummyClassifier(strategy="prior")

def make_logreg():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))

def make_rf():
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE))

def make_xgb():
    # ~42% positives: no imbalance trick needed (scale_pos_weight stays at its default of 1)
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)

MODELS = {"dummy (prior)": make_dummy, "logistic regression": make_logreg,
          "random forest": make_rf, "xgboost": make_xgb}

## 5. Baseline comparison (group split by cartel case)

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)
results, oof_store = [], {}
for fs_name, X in X_sets.items():
    for m_name, mk in MODELS.items():
        if m_name.startswith("dummy") and fs_name != "screens":
            continue
        fold_df, oof = run_cv(mk, X, y, groups, gkf)
        oof_store[(fs_name, m_name)] = oof
        results.append({"features": fs_name, "model": m_name,
                        "pr_auc_mean": fold_df.pr_auc.mean(), "pr_auc_std": fold_df.pr_auc.std(),
                        "roc_auc_mean": fold_df.roc_auc.mean(), "f1_mean": fold_df.f1.mean(),
                        "random_baseline": fold_df.pos_rate.mean()})
res = pd.DataFrame(results).sort_values("pr_auc_mean", ascending=False).reset_index(drop=True)
display(res.round(3))

**How to read this:** a model is only interesting if its PR-AUC is clearly above `random_baseline` *and* above logistic regression.
Look at `pr_auc_std` too: with 73 uneven cases, the score depends heavily on which cartels land in the test folds.

In [ ]:
best = res[~res.model.str.startswith("dummy")].iloc[0]
best_key = (best["features"], best["model"])
Xb = X_sets[best["features"]].reset_index(drop=True)
print("Best (by mean PR-AUC):", best_key)

fold_df, _ = run_cv(MODELS[best["model"]], Xb, y, groups, gkf)
display(fold_df.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(fold_df.fold - 0.2, fold_df.pr_auc, width=0.4, label="PR-AUC")
ax[0].bar(fold_df.fold + 0.2, fold_df.pos_rate, width=0.4, label="random baseline")
ax[0].set_xlabel("fold"); ax[0].set_title("PR-AUC per fold"); ax[0].legend()

oof = oof_store[best_key]
prec, rec, _ = precision_recall_curve(y, oof)
ax[1].plot(rec, prec)
ax[1].axhline(y.mean(), ls="--", color="grey")
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title("Pooled out-of-fold PR curve")
plt.tight_layout(); plt.show()

## 6. How much is leakage? Random split vs group split

A random split puts contracts of the same cartel in both train and test, so the model can recognise the cartel instead of the collusion.
The gap below is what you should explain in the report (the methodology paper shows a similar gap: ~91% accuracy on random contracts vs ~77% on random cartels).

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
rand_df, _ = run_cv(MODELS[best["model"]], Xb, y, groups, skf)
cmp = pd.DataFrame({
    "split": ["random (leaky)", "by cartel case"],
    "pr_auc": [rand_df.pr_auc.mean(), fold_df.pr_auc.mean()],
    "pr_auc_std": [rand_df.pr_auc.std(), fold_df.pr_auc.std()],
    "accuracy": [rand_df.accuracy.mean(), fold_df.accuracy.mean()],
    "f1": [rand_df.f1.mean(), fold_df.f1.mean()],
})
display(cmp.round(3))

## 7. Stress tests

**a) Leave the largest case out.** Largest case = big share of the rows; if the score collapses here, the model leans on that case.
**b) Leave one country out.** Closest to the paper's hardest test. Uses the `screens` set only, since the country dummies would be unseen.

In [ ]:
largest = sizes.index[0]
mask = groups == largest
m = MODELS[best["model"]]().fit(Xb.loc[~mask], y[~mask])
p = m.predict_proba(Xb.loc[mask])[:, 1]
print(f"Largest case: {largest} | rows: {mask.sum()} | positive rate: {y[mask].mean():.3f}")
print(f"PR-AUC: {average_precision_score(y[mask], p):.3f}  (random baseline {y[mask].mean():.3f})")
if len(np.unique(y[mask])) == 2:
    print(f"ROC-AUC: {roc_auc_score(y[mask], p):.3f}")

In [ ]:
country = df["country"].values
lco_df, _ = run_cv(MODELS[best["model"]], X_sets["screens"], y, country, LeaveOneGroupOut())
lco_df = lco_df.rename(columns={"test_groups": "left_out_country"})
display(lco_df[["left_out_country", "n_test", "pos_rate", "pr_auc", "roc_auc", "accuracy"]].round(3))
print("Mean accuracy leaving a country out:", lco_df.accuracy.mean().round(3),
      "| mean PR-AUC:", lco_df.pr_auc.mean().round(3))

## 8. Per-case view

The task is "collusion period vs other contracts of the same firms", so the number that matters is how well the model separates the two **inside** each cartel (`roc_auc_within_case`).
A case with a very high or very low positive rate can look good or bad for the wrong reasons.

In [ ]:
tmp = pd.DataFrame({"case": groups, "y": y, "p": oof_store[best_key]})
case_tbl = tmp.groupby("case").agg(n=("y", "size"), pos_rate=("y", "mean"), mean_pred=("p", "mean"))
within = {c: roc_auc_score(g.y, g.p) for c, g in tmp.groupby("case") if g.y.nunique() == 2}
case_tbl["roc_auc_within_case"] = case_tbl.index.map(within)
display(case_tbl.sort_values("n", ascending=False).head(12).round(3))

## 9. What drives the predictions (exploration only)

Fit on all data **only to look at importances**, never to report performance.
SHAP here is a preview of what the platform will store per contract.

In [ ]:
xgb_full = make_xgb().fit(Xb, y)
imp = pd.Series(xgb_full.feature_importances_, index=Xb.columns).sort_values(ascending=False)
display(imp.head(15).round(3).to_frame("xgb_importance"))

try:
    import shap
    sample = Xb.sample(min(500, len(Xb)), random_state=RANDOM_STATE)
    sv = shap.TreeExplainer(xgb_full).shap_values(sample)
    shap.summary_plot(sv, sample, show=True)
except ImportError:
    print("shap not installed: pip install shap")

## 10. Notes to copy into `docs/eda_findings.md`

Fill these in from the outputs above:

- Strongest screens (overall vs case-weighted): ...
- Best model and features: ... PR-AUC mean ± std = ... vs random baseline ...
- Does xgboost/random forest beat logistic regression? By how much? ...
- Random split vs group split gap: ...
- Largest case left out: ...
- Leave-one-country-out (accuracy / PR-AUC): ...
- Does adding `tender_year` or `country`/sector change the picture? (possible proxy effects) ...
- Cases where the model fails and a hypothesis why: ...

## 11. Next steps (hand-off to the backend)

Turn the notebook into plain functions in `ml_pipeline/` (DataFrame in, DataFrame out), so the backend student can wrap them:

- `build_features(df) -> X` (same banned-column rules)
- `make_model()` (the chosen estimator)
- `cross_validate(df) -> DataFrame` (group split by `cartel_id`)
- `fit_final(df) -> model`, saved to `ml_pipeline/artifacts/`
- `score(df, model) -> DataFrame[contract_id, risk_score, top_features]` (SHAP values)

Then add: threshold selection on out-of-fold predictions, SHAP top-10 per contract, and the `score_contracts` command.